In [ ]:
import shutil
import os

# ========== CONFIGURABLE PATHS ==========
deep_src = os.environ.get("DEEP_SRC_PATH", "/kaggle/input/deep/test_sequence_predictions.csv")
deep_dst = os.environ.get("DEEP_CSV_PATH", "/kaggle/working/deep_predictions.csv")
rf_src   = os.environ.get("RF_SRC_PATH", "/kaggle/input/rf/rf_predictions.csv")
rf_dst   = os.environ.get("RF_CSV_PATH", "/kaggle/working/rf_predictions.csv")
# ========== END CONFIG ==========

shutil.copy(deep_src, deep_dst)
print(f"Copied {deep_src} to {deep_dst}")
shutil.copy(rf_src, rf_dst)
print(f"Copied {rf_src} to {rf_dst}")

In [ ]:
#!/usr/bin/env python3

import os
import re
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- User params (set these) ---------
deep_csv_path = os.environ.get("DEEP_CSV_PATH", "/kaggle/working/deep_predictions.csv")   # path to deep CSV 
rf_csv_path   = os.environ.get("RF_CSV_PATH", "/kaggle/working/rf_predictions.csv")     # path to RF CSV 
merge_on = "sequence_path"  # column name in both CSVs that contains the path-like value
out_merged_csv = os.environ.get("MERGED_CSV", "/kaggle/working/merged_deep_rf_by_last3.csv")

# ---------- Normalization helpers ----------
def last3_dir_of_path(p):
    if pd.isna(p):
        return ""
    s = str(p).replace("\\", "/").strip()
    s = re.sub(r'^[a-zA-Z]+://', '', s)             
    parts = [pt.strip() for pt in s.split("/") if pt.strip() != ""]
    # if last element looks like a filename (contains a dot), drop it
    if parts and "." in parts[-1] and len(parts[-1].split(".")) <= 3:
        parts = parts[:-1]
    if len(parts) >= 3:
        return "/".join(parts[-3:])
    else:
        return "/".join(parts)

def normalize_key(p):
    k = last3_dir_of_path(p)
    k = re.sub(r"\s+", " ", k).strip().lower()
    return k

# ---------- Load CSVs ----------
if not os.path.exists(deep_csv_path):
    raise FileNotFoundError(f"Deep CSV not found: {deep_csv_path}")
if not os.path.exists(rf_csv_path):
    raise FileNotFoundError(f"RF CSV not found: {rf_csv_path}")

deep_df = pd.read_csv(deep_csv_path)
rf_df   = pd.read_csv(rf_csv_path)
print(f"Deep CSV rows: {len(deep_df)}  RF CSV rows: {len(rf_df)}")
print("Preferred merge key:", merge_on)

# If merge_on not present in either CSV, try to pick a path-like column
def detect_path_col(df):
    candidates = list(df.columns)
    for c in candidates:
        sample = df[c].dropna().astype(str).head(200).tolist()
        if any(re.search(r'S\d{3}[/\\]', s, flags=re.IGNORECASE) for s in sample):
            return c
    return None

if merge_on not in deep_df.columns or merge_on not in rf_df.columns:
    deep_col = merge_on if merge_on in deep_df.columns else detect_path_col(deep_df)
    rf_col   = merge_on if merge_on in rf_df.columns else detect_path_col(rf_df)
    if deep_col is None or rf_col is None:
        print("Could not find a path-like column in one of the CSVs.")
        print("Deep columns:", list(deep_df.columns))
        print("RF   columns:", list(rf_df.columns))
        raise RuntimeError("Please set 'merge_on' to an existing column name in both CSVs or ensure each CSV has a path-like column.")
    print(f"Using deep key column '{deep_col}' and rf key column '{rf_col}' for normalization.")
    deep_key_col, rf_key_col = deep_col, rf_col
else:
    deep_key_col = rf_key_col = merge_on
    print(f"Using '{merge_on}' from both CSVs as key column.")

# ---------- Create normalized keys ----------
deep_df["_raw_key"] = deep_df[deep_key_col].astype(str)
rf_df["_raw_key"]   = rf_df[rf_key_col].astype(str)

deep_df["_key_last3"] = deep_df["_raw_key"].map(normalize_key)
rf_df["_key_last3"]   = rf_df["_raw_key"].map(normalize_key)

# Diagnostics: show samples
print("\nExample (deep) source keys (head 10):")
print(deep_df[["_raw_key", "_key_last3"]].head(10).to_string(index=False))
print("\nExample (rf) source keys (head 10):")
print(rf_df[["_raw_key", "_key_last3"]].head(10).to_string(index=False))

# Intersection / mismatch diagnostics
deep_keys = set(k for k in deep_df["_key_last3"].unique() if k)
rf_keys   = set(k for k in rf_df["_key_last3"].unique() if k)
intersection = deep_keys & rf_keys
print(f"\nUnique deep last3 keys: {len(deep_keys)}")
print(f"Unique rf   last3 keys: {len(rf_keys)}")
print(f"Intersection: {len(intersection)}")

left_key = right_key = "_key_last3"
# If no overlap, try alternate normalizations
if len(intersection) == 0:
    print("\nNo overlap found using normalized last-3 keys. Trying fallbacks...")

    # fallback 1: suffix-2 
    def suffix2(p):
        k = last3_dir_of_path(p)
        parts = [pt for pt in k.split("/") if pt != ""]
        if len(parts) >= 2:
            return "/".join(parts[-2:]).strip().lower()
        return k.lower()

    deep_df["_key_suffix2"] = deep_df["_raw_key"].map(suffix2)
    rf_df["_key_suffix2"]   = rf_df["_raw_key"].map(suffix2)
    inter_suf2 = set(k for k in deep_df["_key_suffix2"].unique() if k) & set(k for k in rf_df["_key_suffix2"].unique() if k)
    print("Suffix-2 intersection size:", len(inter_suf2))
    if len(inter_suf2) > 0:
        left_key = right_key = "_key_suffix2"
    else:
        # fallback 2: alt normalization 
        def normalize_alt(p):
            k = last3_dir_of_path(p)
            k = re.sub(r"(bw|colour|colour oval|bw oval|oval|ovalo|ovalo bn)", lambda m: m.group(0).replace(" ", " "), k, flags=re.IGNORECASE)
            k = re.sub(r"\s+", " ", k).strip().lower()
            parts = [pt for pt in k.split("/") if pt != ""]
            if len(parts) >= 3 and parts[-1] == parts[-2]:
                parts = parts[:-1]
            return "/".join(parts)
        deep_df["_key_alt"] = deep_df["_raw_key"].map(normalize_alt)
        rf_df["_key_alt"]   = rf_df["_raw_key"].map(normalize_alt)
        inter_alt = set(k for k in deep_df["_key_alt"].unique() if k) & set(k for k in rf_df["_key_alt"].unique() if k)
        print("Alternate normalization intersection size:", len(inter_alt))
        if len(inter_alt) > 0:
            left_key = right_key = "_key_alt"
        else:
            # final diagnostics and informative error
            print("\nSamples (deep normalized last3) examples (up to 20):")
            print(sorted(list(deep_keys))[:20])
            print("\nSamples (rf normalized last3) examples (up to 20):")
            print(sorted(list(rf_keys))[:20])
            raise RuntimeError("No overlapping rows found after normalization attempts. Inspect the sample lists above.")

else:
    print("Sufficient overlap found using last-3 normalization.")

# ---------- Perform merge using chosen key ----------
print(f"\nMerging using keys: deep->{left_key} rf->{right_key}")
merged = pd.merge(
    deep_df,
    rf_df,
    left_on=left_key,
    right_on=right_key,
    how="inner",
    suffixes=("_deep", "_rf")
)

print("Merged rows (inner join):", len(merged))
if merged.empty:
    raise RuntimeError("Merge produced 0 rows despite earlier intersection check. Check key columns and uniqueness.")

# Canonical true_label selection 
if "true_label_deep" in merged.columns:
    merged["true_label"] = merged["true_label_deep"].astype(int)
elif "true_label" in merged.columns and "true_label_deep" not in merged.columns and "true_label_rf" not in merged.columns:
    merged["true_label"] = merged["true_label"].astype(int)
elif "true_label_rf" in merged.columns:
    merged["true_label"] = merged["true_label_rf"].astype(int)
else:
    raise RuntimeError("No true_label column available after merge")

# Save merged for inspection
out_dir = Path(out_merged_csv).parent
out_dir.mkdir(parents=True, exist_ok=True)
merged.to_csv(out_merged_csv, index=False)
print("Saved merged CSV to:", out_merged_csv)

# ---------- Safe preview (avoid KeyError) ----------
print("\nMerged columns:\n", merged.columns.tolist())

display_key = None
for cand in [merge_on, f"{merge_on}_deep", f"{merge_on}_rf", "_raw_key_deep", "_raw_key", left_key, right_key]:
    if cand in merged.columns:
        display_key = cand
        break
if display_key is None:
    for c in merged.columns:
        sample_vals = merged[c].astype(str).head(50).tolist()
        if any(re.search(r's\d{3}[/\\]', str(v), flags=re.IGNORECASE) for v in sample_vals):
            display_key = c
            break
if display_key is None:
    display_key = merged.columns[0]

print("Using display_key for preview:", display_key)

show_cols = [c for c in [display_key, left_key, right_key, "true_label", "predicted_label_deep", "prob_pain_deep", "predicted_label_rf", "prob_pain_rf", "predicted_label"] if c in merged.columns]
if not show_cols:
    show_cols = merged.columns[:8].tolist()

print("\nPreview of merged rows (head 10):")
print(merged[show_cols].head(10).to_string(index=False))

# done
print("\nDone. You can now run fusion on `merged` DataFrame (e.g., average deep/rf prob_pain) and produce a trimmed CSV without raw path columns.")

In [ ]:
#!/usr/bin/env python3
"""
This cell is for computing late-fusion on a merged deep+handcrafted DataFrame (or merged CSV),
produce a trimmed fused CSV (no long path column), and print metrics
(accuracy, precision, recall, F1, confusion matrix, classification report).

"""
import os
import re
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# ---------- Settings ----------
# Path to the merged CSV produced by the merge/normalize script 
out_merged_csv = os.environ.get("MERGED_CSV", "/kaggle/working/merged_deep_rf_by_last3.csv")
out_trimmed_csv = os.environ.get("FUSED_CSV", "/kaggle/working/fused_trimmed.csv")

fusion_method = "weighted"
#assign weights
weight_deep = 0.40  # change deep weight
weight_rf = 0.60  # cgange random forrest weight

# If using stacking, number of CV folds for OOF stacking (must be >=2)
stack_cv_folds = 5

keep_source_probs = True

# ---------- Helper utilities ----------
def try_load_merged():
    if 'merged' in globals():
        return globals()['merged']
    if os.path.exists(out_merged_csv):
        df = pd.read_csv(out_merged_csv)
        print(f"Loaded merged CSV from: {out_merged_csv} (rows={len(df)})")
        return df
    raise FileNotFoundError("Merged DataFrame not found in memory and merged CSV not found at: " + out_merged_csv)

def find_prob_col(df, hint):
    # Try common patterns for prob_pain for each side
    candidates = [
        f"{hint}_prob_pain", f"prob_pain_{hint}", f"prob_{hint}_pain",
        f"prob_pain_{hint}".lower(), "prob_pain", "prob_1", "prob1"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if "prob" in lc and "pain" in lc:
            return c
    return None

def find_pred_col(df, hint):
    candidates = [f"{hint}_pred", f"{hint}_predicted_label", "predicted_label", "pred_label", "pred"]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if ("pred" in lc or "label" in lc) and "true" not in lc:
            return c
    return None

# ---------- Fusion helpers ----------
def fuse_avg_row(r):
    p = 0.5 * (r["deep_prob_pain"] + r["rf_prob_pain"])
    return p, 1.0 - p

def fuse_weighted_row(r, w_d, w_r):
    p = w_d * r["deep_prob_pain"] + w_r * r["rf_prob_pain"]
    return p, 1.0 - p

def fuse_vote_row(r):
    votes = [int(r["deep_pred"]), int(r["rf_pred"])]
    maj = int(round(np.mean(votes)))
    if sum(votes) * 2 == len(votes):  # tie
        p = 0.5 * (r["deep_prob_pain"] + r["rf_prob_pain"])
        return p, 1.0 - p
    return float(maj), float(1 - maj)

def fuse_confidence_select_row(r):
    dconf = max(r["deep_prob_pain"], r.get("deep_prob_nopain", 1.0 - r["deep_prob_pain"]))
    rconf = max(r["rf_prob_pain"], r.get("rf_prob_nopain", 1.0 - r["rf_prob_pain"]))
    if dconf >= rconf:
        return r["deep_prob_pain"], r.get("deep_prob_nopain", 1.0 - r["deep_prob_pain"])
    return r["rf_prob_pain"], r.get("rf_prob_nopain", 1.0 - r["rf_prob_pain"])

def stack_lr_probs(df, cv_folds=5):
    X = df[["deep_prob_pain", "rf_prob_pain"]].values
    y = df["true_label"].astype(int).values
    oof = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=min(cv_folds, max(2, int(np.sum(y==0))+1)), shuffle=True, random_state=42)
    for train_idx, val_idx in skf.split(X, y):
        clf = LogisticRegression(max_iter=500)
        clf.fit(X[train_idx], y[train_idx])
        oof[val_idx] = clf.predict_proba(X[val_idx])[:, 1]
    final = LogisticRegression(max_iter=500).fit(X, y)
    final_probs = final.predict_proba(X)[:, 1]
    return final_probs, oof, final

# ---------- Run fusion ----------
# Load merged DataFrame (from memory or CSV)
df = try_load_merged()

true_col = None
for c in ["true_label", "true_label_deep", "true_label_rf"]:
    if c in df.columns:
        true_col = c
        break
if true_col is None:
    raise RuntimeError("true_label column not found in merged DataFrame. Columns: " + ", ".join(df.columns))

# Detect deep and rf probability columns
deep_prob_col = find_prob_col(df, "deep")
rf_prob_col = find_prob_col(df, "rf")
# fallbacks if the merged columns are suffixed like prob_pain_deep / prob_pain_rf
if deep_prob_col is None:
    for c in df.columns:
        if c.lower().endswith("_deep") and "prob" in c.lower():
            deep_prob_col = c; break
if rf_prob_col is None:
    for c in df.columns:
        if c.lower().endswith("_rf") and "prob" in c.lower():
            rf_prob_col = c; break

if deep_prob_col is None or rf_prob_col is None:
    raise RuntimeError("Could not detect both deep/rf probability columns. Columns: " + ", ".join(df.columns))

# Canonicalize columns
df["deep_prob_pain"] = df[deep_prob_col].astype(float)
df["rf_prob_pain"] = df[rf_prob_col].astype(float)
df["deep_prob_nopain"] = 1.0 - df["deep_prob_pain"]
df["rf_prob_nopain"] = 1.0 - df["rf_prob_pain"]

# Detect/construct predicted label columns for each side
deep_pred_col = find_pred_col(df, "deep")
rf_pred_col = find_pred_col(df, "rf")
if deep_pred_col and deep_pred_col in df.columns:
    df["deep_pred"] = df[deep_pred_col].astype(int)
else:
    df["deep_pred"] = (df["deep_prob_pain"] >= 0.5).astype(int)
if rf_pred_col and rf_pred_col in df.columns:
    df["rf_pred"] = df[rf_pred_col].astype(int)
else:
    df["rf_pred"] = (df["rf_prob_pain"] >= 0.5).astype(int)

# Ensure canonical true_label column
df["true_label"] = df[true_col].astype(int)


if fusion_method == "avg":
    fused = df.apply(lambda r: fuse_avg_row(r), axis=1)
    fused_arr = np.array(list(fused))
    df["prob_pain"] = fused_arr[:,0]

elif fusion_method == "weighted":
    # If explicit weights were provided in the Settings section, use them.
    # Otherwise infer weights from each model's accuracy on the merged rows (not recommended on true test data).
    if weight_deep is None or weight_rf is None:
        acc_deep = accuracy_score(df["true_label"], df["deep_pred"])
        acc_rf = accuracy_score(df["true_label"], df["rf_pred"])
        if acc_deep + acc_rf == 0:
            w_d, w_r = 0.5, 0.5
        else:
            # infer weight proportionally to accuracy
            w_d = acc_deep / (acc_deep + acc_rf)
            w_r = acc_rf / (acc_deep + acc_rf)
        print(f"Inferred weights from labels: deep={w_d:.3f}, rf={w_r:.3f}")
    else:
        # Use the user-provided weights (could be percentages or fractions)
        w_d = float(weight_deep)
        w_r = float(weight_rf)
        print(f"Using provided weights (before normalization): deep={w_d}, rf={w_r}")

    # Normalize weights so they sum to 1. This allows values like 60 and 40 to be used directly.
    w_sum = float(w_d + w_r)
    if w_sum == 0:
        # safety fallback
        w_d_norm = w_r_norm = 0.5
    else:
        w_d_norm = w_d / w_sum
        w_r_norm = w_r / w_sum

    # Debug print to confirm weights used
    print(f"Normalized weights -> deep: {w_d_norm:.3f}, rf: {w_r_norm:.3f}")

    # Apply weighted fusion row-wise
    fused = df.apply(lambda r: fuse_weighted_row(r, w_d_norm, w_r_norm), axis=1)
    fused_arr = np.array(list(fused))
    df["prob_pain"] = fused_arr[:,0]

elif fusion_method == "vote":
    fused = df.apply(lambda r: fuse_vote_row(r), axis=1)
    fused_arr = np.array(list(fused))
    df["prob_pain"] = fused_arr[:,0]

elif fusion_method == "confidence_select":
    fused = df.apply(lambda r: fuse_confidence_select_row(r), axis=1)
    fused_arr = np.array(list(fused))
    df["prob_pain"] = fused_arr[:,0]

elif fusion_method == "stack_lr":
    final_probs, oof, model = stack_lr_probs(df, cv_folds=stack_cv_folds)
    df["prob_pain"] = final_probs

else:
    raise ValueError("Unknown fusion_method: " + str(fusion_method))

# Finalize fused outputs
df["prob_nopain"] = 1.0 - df["prob_pain"]
df["predicted_label"] = (df["prob_pain"] >= 0.5).astype(int)

# ---------- Metrics ----------
y_true = df["true_label"].astype(int).values
y_pred = df["predicted_label"].astype(int).values
acc = accuracy_score(y_true, y_pred)
prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
prec_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
rec_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nFused results (method={}):".format(fusion_method))
print(f"  Rows: {len(df)}")
print(f"  Accuracy: {acc:.4f}")
print(f"  Precision macro: {prec_macro:.4f}  weighted: {prec_weighted:.4f}")
print(f"  Recall    macro: {rec_macro:.4f}  weighted: {rec_weighted:.4f}")
print(f"  F1        macro: {f1_macro:.4f}  weighted: {f1_weighted:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["Neutral", "Pain"]))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

# ---------- Build trimmed output ----------
# Attempt to find normalized key column to parse subject/stim/seq
norm_key_candidates = [c for c in df.columns if c in ("_key_last3","_key_alt","_key_suffix2","sequence_path","sequence_path_norm")]
norm_key = norm_key_candidates[0] if norm_key_candidates else None
if norm_key is None:
    norm_key = None
    for c in df.columns:
        sample = df[c].astype(str).head(100).tolist()
        if any(re.search(r's\d{3}[/\\]', str(s), flags=re.IGNORECASE) for s in sample):
            norm_key = c
            break

if norm_key is None:
    trimmed = df[["true_label","predicted_label","prob_pain","prob_nopain"]].copy()
    if keep_source_probs:
        trimmed["deep_prob_pain"] = df["deep_prob_pain"]
        trimmed["rf_prob_pain"] = df["rf_prob_pain"]
else:
    def parse_from_key(s):
        s = str(s).replace("\\","/").strip()
        parts = [p for p in s.split("/") if p != ""]
        if len(parts) >= 3:
            return parts[-3], parts[-2], parts[-1]
        elif len(parts) == 2:
            return parts[-2], parts[-1], ""
        elif len(parts) == 1:
            return parts[-1], "", ""
        else:
            return "", "", ""

    parsed = df[norm_key].astype(str).map(lambda p: pd.Series(parse_from_key(p)))
    parsed.columns = ["subject_id","stimulus_type","sequence_name"]
    trimmed = pd.concat([parsed.reset_index(drop=True), df[["true_label","predicted_label","prob_pain","prob_nopain","deep_prob_pain","rf_prob_pain"]].reset_index(drop=True)], axis=1)
    if not keep_source_probs and "deep_prob_pain" in trimmed.columns:
        trimmed = trimmed.drop(columns=["deep_prob_pain","rf_prob_pain"], errors="ignore")

# Save trimmed CSV 
trimmed.to_csv(out_trimmed_csv, index=False)
print(f"\nSaved trimmed fused CSV to: {out_trimmed_csv}")
print("\nTrimmed preview:")
print(trimmed.head(12).to_string(index=False))